In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from scipy.stats import gaussian_kde
from scipy.signal import argrelmax
from nilearn.maskers import NiftiMasker

from abstract_values.utils.data import Subject, BIDS_FOLDER

In [ ]:
SUBJECTS = ['pil01', 'pil02']
R2_THR   = 0.05
VALUE_MIN, VALUE_MAX = 2.5, 41.5

# ── value distributions ────────────────────────────────────────────────────
# 25 orientations (0–180° in 7.5° steps) each map to one CHF value.
# Densely packed values = many orientations per CHF unit = high density.
CDF_VALUES     = np.array([2.0, 5.5, 8.0, 10.0, 11.0, 11.5, 12.0, 12.5,
                            13.0, 14.0, 16.0, 18.5, 22.0, 25.5, 28.0, 30.0,
                            31.0, 31.5, 32.0, 32.5, 33.0, 34.0, 36.0, 38.5, 42.0])
INV_CDF_VALUES = np.array([2.0, 2.5, 3.0, 4.0, 5.0, 7.5, 11.5, 16.0, 18.5,
                            20.0, 21.0, 21.5, 22.0, 22.5, 23.0, 24.0, 25.5,
                            28.0, 32.5, 36.5, 39.0, 40.0, 41.0, 41.5, 42.0])

# CDF is bimodal (dense at 11–13 and 31–33 CHF → peaks ~12 and ~32 CHF).
# inv_CDF is unimodal (dense at 21–23 CHF → peak ~22 CHF).
MAPPING_PEAKS = {
    'cdf':     [12.0, 32.0],
    'inv_cdf': [22.0],
}

def get_session_mapping(subject_label, session):
    """Return mapping name ('cdf' or 'inv_cdf') for this pilot subject + session.

    Even pilot number → ses-1 = cdf,     ses-2 = inv_cdf
    Odd  pilot number → ses-1 = inv_cdf, ses-2 = cdf
    """
    pilot_num = int(''.join(c for c in subject_label if c.isdigit()))
    is_even = (pilot_num % 2 == 0)
    if (is_even and session == 1) or (not is_even and session == 2):
        return 'cdf'
    else:
        return 'inv_cdf'


for s in SUBJECTS:
    m1 = get_session_mapping(s, 1)
    m2 = get_session_mapping(s, 2)
    print(f'{s}:  ses-1 = {m1}  |  ses-2 = {m2}')

In [ ]:
def load_subject_data(subject, r2_thr=R2_THR):
    sub = Subject(subject, bids_folder=BIDS_FOLDER)
    out_dir = (BIDS_FOLDER / 'derivatives' / 'encoding_models'
               / 'aprf-session-shift' / f'sub-{subject}' / 'func')

    def load(desc):
        return out_dir / f'sub-{subject}_task-abstractvalue_space-T1w_desc-{desc}_pe.nii.gz'

    masker = NiftiMasker(mask_img=sub.get_roi_mask('NPCr', hemi=None)).fit()
    r2    = masker.transform(load('r2')).squeeze()
    mode1 = masker.transform(load('mode_1')).squeeze()
    mode2 = masker.transform(load('mode_2')).squeeze()

    sel = r2 > r2_thr
    print(f'{subject}: {sel.sum():,} / {len(r2):,} voxels above R²={r2_thr}')
    return dict(
        r2=r2, mode1=mode1, mode2=mode2, sel=sel,
        map1=get_session_mapping(subject, 1),
        map2=get_session_mapping(subject, 2),
    )


data = {s: load_subject_data(s) for s in SUBJECTS}

In [ ]:
# ── joint mode plot ────────────────────────────────────────────────────────
# Vertical lines = x-axis (ses-1) distribution peaks
# Horizontal lines = y-axis (ses-2) distribution peaks

PEAK_COLORS = {'cdf': 'cyan', 'inv_cdf': 'orange'}

fig, axes = plt.subplots(1, len(SUBJECTS), figsize=(6 * len(SUBJECTS), 6),
                          constrained_layout=True)
if len(SUBJECTS) == 1:
    axes = [axes]

for ax, subject in zip(axes, SUBJECTS):
    d = data[subject]
    m1 = d['mode1'][d['sel']]
    m2 = d['mode2'][d['sel']]
    map1, map2 = d['map1'], d['map2']

    hb = ax.hexbin(m1, m2, gridsize=40, cmap='viridis',
                   extent=[VALUE_MIN, VALUE_MAX, VALUE_MIN, VALUE_MAX],
                   mincnt=1)

    # identity line
    ax.plot([VALUE_MIN, VALUE_MAX], [VALUE_MIN, VALUE_MAX],
            'w--', lw=1, alpha=0.5)

    # ses-1 distribution peaks → vertical lines
    for i, px in enumerate(MAPPING_PEAKS[map1]):
        label = f'ses-1 ({map1}) peak: {px:.0f} CHF' if i == 0 else None
        ax.axvline(px, color=PEAK_COLORS[map1], lw=1.5, ls=':', alpha=0.9, label=label)

    # ses-2 distribution peaks → horizontal lines
    for i, py in enumerate(MAPPING_PEAKS[map2]):
        label = f'ses-2 ({map2}) peak: {py:.0f} CHF' if i == 0 else None
        ax.axhline(py, color=PEAK_COLORS[map2], lw=1.5, ls='--', alpha=0.9, label=label)

    ax.set_xlabel(f'Mode session 1 ({map1}) [CHF]', fontsize=12)
    ax.set_ylabel(f'Mode session 2 ({map2}) [CHF]', fontsize=12)
    ax.set_title(f'sub-{subject}  NPCr  (R²>{R2_THR}, n={d["sel"].sum():,})',
                 fontsize=11)
    ax.set_xlim(VALUE_MIN, VALUE_MAX)
    ax.set_ylim(VALUE_MIN, VALUE_MAX)
    ax.set_aspect('equal')
    ax.legend(fontsize=8, loc='upper left')
    plt.colorbar(hb, ax=ax, label='voxel count', shrink=0.8)

fig.suptitle('Session-shift aPRF: preferred value per session', fontsize=13)
plt.show()

In [ ]:
# ── delta-mode histograms ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(SUBJECTS), figsize=(6 * len(SUBJECTS), 4),
                          constrained_layout=True)
if len(SUBJECTS) == 1:
    axes = [axes]

for ax, subject in zip(axes, SUBJECTS):
    d = data[subject]
    m1 = d['mode1'][d['sel']]
    m2 = d['mode2'][d['sel']]
    delta = m2 - m1
    map1, map2 = d['map1'], d['map2']

    ax.hist(delta, bins=60, color='steelblue', edgecolor='none')
    ax.axvline(0, color='k', lw=1.5, ls='--')
    ax.axvline(delta.mean(), color='tomato', lw=1.5,
               label=f'mean = {delta.mean():.2f} CHF')

    ax.set_xlabel(f'mode₂ ({map2}) − mode₁ ({map1}) [CHF]', fontsize=12)
    ax.set_ylabel('voxel count', fontsize=12)
    ax.set_title(f'sub-{subject}  shift in preferred value', fontsize=11)
    ax.legend()

plt.show()

In [ ]:
# ── Fisher information per session (session-shift aPRF model) ──────────────
N_VOXELS  = 100
ss_deriv  = BIDS_FOLDER / 'derivatives' / 'encoding_models' / 'aprf-session-shift'

ses_colors = {1: 'steelblue', 2: 'tomato'}

fig, axes = plt.subplots(1, len(SUBJECTS), figsize=(7 * len(SUBJECTS), 4),
                          constrained_layout=True)
if len(SUBJECTS) == 1:
    axes = [axes]

for ax, subject in zip(axes, SUBJECTS):
    d = data[subject]
    found_any = False
    for ses in [1, 2]:
        mapping = get_session_mapping(subject, ses)
        fn = (ss_deriv / f'sub-{subject}' / f'ses-{ses}' / 'func'
              / f'sub-{subject}_ses-{ses}_task-abstractvalue'
                f'_mask-NPCr_nvoxels-{N_VOXELS}_desc-fisherinfo_pe.tsv')
        if not fn.exists():
            print(f'  {subject} ses-{ses}: FI file not found, skipping')
            continue
        fi = pd.read_csv(fn, sep='\t', index_col=0).squeeze()
        ax.plot(fi.index, fi.values,
                color=ses_colors[ses], lw=2,
                label=f'ses-{ses} ({mapping})')
        peak_x = fi.index[fi.values.argmax()]
        ax.axvline(peak_x, color=ses_colors[ses], lw=1, ls='--', alpha=0.6)
        print(f'  {subject} ses-{ses} ({mapping}): peak FI at {peak_x:.1f} CHF')
        found_any = True

    if not found_any:
        ax.text(0.5, 0.5, 'No FI data available', transform=ax.transAxes,
                ha='center', va='center', fontsize=12, color='gray')

    ax.set_xlabel('Value [CHF]', fontsize=12)
    ax.set_ylabel('Fisher information', fontsize=12)
    ax.set_title(f'sub-{subject}  session-shift aPRF  FI  (top-{N_VOXELS} NPCr)',
                 fontsize=11)
    ax.legend()

plt.show()